# AI Code Review RL Environment Sandbox

Interactive exploration of the reinforcement learning environment for code review task.

## Setup

In [ ]:
import sys
sys.path.insert(0, '.')

from app.environment import CodeReviewEnv
from app.models.schemas import Action, ActionType
import json
from pprint import pprint

## Initialize Environment

In [ ]:
env = CodeReviewEnv()
print("✓ Environment initialized")

## View Available Tasks

In [ ]:
from app.tasks.task_definitions import ALL_TASKS

print(f"Available tasks: {len(ALL_TASKS)}")
for task in ALL_TASKS:
    print(f"\n📋 {task['name']} ({task['difficulty']})")
    print(f"   Description: {task['description']}")
    print(f"   Expected issues: {len(task['issues'])}")

## Reset Environment & Start Episode

In [ ]:
# Reset with a specific task or None to cycle automatically
reset_response = env.reset(task_name=None)

print("🔄 Episode reset")
print(f"Task: {reset_response.task_info.name}")
print(f"Difficulty: {reset_response.task_info.difficulty}")
print(f"Code snippet:\n{reset_response.observation.code}")

## Inspect Current State

In [ ]:
state = env.state()

print("📊 Current State:")
print(f"Step: {state.step_count}")
print(f"Total Reward: {state.total_reward}")
print(f"Done: {state.done}")
print(f"Code snippet length: {len(state.observation.code)} chars")
print(f"Expected issues to find: {state.task_info.expected_issues}")

## Take Actions

Available action types:
- `FLAG_ISSUE`: Flag a bug at a specific line
- `SUGGEST_FIX`: Suggest a fix for an issue
- `APPROVE`: Approve the code as correct

In [ ]:
# Example: Flag an issue
action = Action(
    action_type=ActionType.FLAG_ISSUE,
    issue_type="LOGIC_ERROR",
    line_number=5,
    description="Off-by-one error in loop condition",
    suggested_fix=None
)

result = env.step(action)

print("✅ Action executed")
print(f"Reward: {result.reward}")
print(f"Done: {result.done}")
print(f"Total Reward: {result.total_reward}")
if result.info:
    print(f"Info: {result.info}")

## Take More Actions

In [ ]:
# Example: Suggest a fix for the flagged issue
action = Action(
    action_type=ActionType.SUGGEST_FIX,
    issue_type="LOGIC_ERROR",
    line_number=5,
    description="Change < to <=",
    suggested_fix="for i in range(len(items) + 1):"
)

result = env.step(action)

print("✅ Fix suggested")
print(f"Reward: {result.reward}")
print(f"Total Reward: {result.total_reward}")
print(f"Done: {result.done}")

## Approve Code

In [ ]:
# Approve the code
action = Action(
    action_type=ActionType.APPROVE,
    issue_type=None,
    line_number=None,
    description="Code approved after fixes",
    suggested_fix=None
)

result = env.step(action)

print("✅ Code approved")
print(f"Reward: {result.reward}")
print(f"Total Reward: {result.total_reward}")
print(f"Episode Done: {result.done}")

## Get Final Score

In [ ]:
from app.tasks.task_definitions import grade_episode

# Get the current state for grading
final_state = env.state()

print("🎯 Final Episode Summary:")
print(f"Total Steps: {final_state.step_count}")
print(f"Total Reward: {final_state.total_reward}")
print(f"Expected Issues: {final_state.task_info.expected_issues}")
print(f"Episode Done: {final_state.done}")

## Run Multiple Episodes

In [ ]:
# Quick test: run 3 simple episodes
for episode in range(3):
    print(f"\n--- Episode {episode + 1} ---")
    reset_resp = env.reset()
    print(f"Task: {reset_resp.task_info.name} ({reset_resp.task_info.difficulty})")
    
    # Take one action (approve immediately as baseline)
    action = Action(
        action_type=ActionType.APPROVE,
        issue_type=None,
        line_number=None,
        description="Baseline approval",
        suggested_fix=None
    )
    result = env.step(action)
    print(f"Reward: {result.reward}, Total Reward: {result.total_reward}")